# Импорт библиотек

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from joblib import dump, load
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score
from joblib import Parallel, delayed

In [2]:
X_train_taxi = pd.read_csv('/home/jupyter/datasets/nyc-taxi-small-public/X_train_public.csv')
train_fraud = pd.read_csv('/home/jupyter/datasets/ieee-fraud-detection/train_public.csv')

/tmp/ipykernel_8439/2701212499.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  X_train_taxi = pd.read_csv('/home/jupyter/datasets/nyc-taxi-small-public/X_train_public.csv')


# Кастомные трансформеры

In [3]:
class DateTimeFeatureGenerator(BaseEstimator, TransformerMixin):
    """Преобразуем datetime-столбец в набор признаков: час, день, день недели и месяц."""
    
    def __init__(self, column):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        df = pd.to_datetime(X[self.column]) # преобразуем столбец в формат datetime
        
        # генерируем новые признаки из даты и времени
        X["pickup_hour"] = df.dt.hour
        X["pickup_day"] = df.dt.day
        X["pickup_weekday"] = df.dt.weekday
        X["pickup_month"] = df.dt.month

        X = X.drop(columns=[self.column]) # удаляем исходный datetime-столбец

        return X

In [4]:
class FraudFeatureGenerator(BaseEstimator, TransformerMixin):
    """Создание специфичных признаков для обнаружения мошенничества"""
    
    def __init__(self):
        self.columns_to_drop_ = ["TransactionID"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        # Признаки на основе TransactionDT (время)
        if "TransactionDT" in X.columns:
            # TransactionDT — число секунд от некоторой точки отсчёта
            X["TransactionDT_days"] = X["TransactionDT"] // (24 * 60 * 60)
            X["TransactionDT_hours"] = (X["TransactionDT"] // (60 * 60)) % 24
            X["TransactionDT_week"] = X["TransactionDT"] // (7 * 24 * 60 * 60)

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])

        return X

In [5]:
class IQRToNaN(BaseEstimator, TransformerMixin):
    """Замена выбросов в числовых признаках на NaN по правилу IQR."""
    
    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):

        self.num_cols_ = X.select_dtypes(include=np.number).columns # выбираем только числовые столбцы
        
        # Считаем первый и третий квартили для числовых признаков
        q1 = X[self.num_cols_].quantile(0.25)
        q3 = X[self.num_cols_].quantile(0.75)
        iqr = q3 - q1 # межквартильный размах

        # Вычисляем нижнюю и верхнюю границы допустимых значений
        self.lower_bounds_ = q1 - self.factor * iqr
        self.upper_bounds_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        X = X.copy()
        # Во всех числовых столбцах заменяем значения, выходящие за границы IQR, на NaN
        X[self.num_cols_] = X[self.num_cols_].mask((X[self.num_cols_] < self.lower_bounds_) | (X[self.num_cols_] > self.upper_bounds_))
        
        return X

In [36]:
# Загрузка пайплайнов из предыдущих работ
numeric_pipeline = load('numeric_pipeline.joblib')
categorical_pipeline = load('categorical_pipeline.joblib')

# Выделение псевдотаргетов

In [22]:
# выделяем псевдо-таргет
pseudo_target_taxi = X_train_taxi['payment_type']

In [23]:
# выделяем псевдо-таргет
pseudo_target_fraud = train_fraud['card4']

# Выделение матриц признаков 

NYC Taxi - получение X

In [24]:
X_train_taxi_copy = X_train_taxi.drop(columns=['payment_type'])

In [25]:
num_cols_taxi = X_train_taxi_copy.select_dtypes(include=["int64","float64"]).columns
cat_cols_taxi = X_train_taxi_copy.select_dtypes(include=["object"]).columns

In [26]:
datetime_col = "tpep_pickup_datetime"

num_cols_taxi = num_cols_taxi.drop(datetime_col, errors="ignore")
cat_cols_taxi = cat_cols_taxi.drop(datetime_col, errors="ignore")

In [27]:
taxi_pipeline = Pipeline([
    ("datetime", DateTimeFeatureGenerator("tpep_pickup_datetime")),
    ("preprocessor", ColumnTransformer([
        ("num", numeric_pipeline, num_cols_taxi),
        ("cat", categorical_pipeline, cat_cols_taxi)
    ])),
])

In [28]:
X_taxi = taxi_pipeline.fit_transform(X_train_taxi_copy)

IEEE Fraud - получение X

In [37]:
X_train_fraud = train_fraud.drop(columns=['card4', 'isFraud'])

In [38]:
numerical_features_fraud = X_train_fraud.select_dtypes(include=["int64","float64"]).columns
categorical_features_fraud = X_train_fraud.select_dtypes(include=["object"]).columns

In [39]:
fraud_preprocessing = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features_fraud),
    ("cat", categorical_pipeline, categorical_features_fraud)
])

In [40]:
fraud_pipeline = Pipeline([
    ("feature_generator", FraudFeatureGenerator()),
    ("preprocessor", fraud_preprocessing)
])

In [41]:
X_fraud = fraud_pipeline.fit_transform(X_train_fraud)

# Алгоритмы кластеризации

In [42]:
class MyKMeans:
    """Реализация алгоритма k-means"""
    def __init__(self, n_clusters=5, max_iter=50, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters # кол-во кластеров
        self.max_iter = max_iter # максимальное кол-во итераций
        self.tol = tol # порог сходимости
        self.random_state = random_state # "зерно" генератора случайных чисел

    def fit(self, X):
        np.random.seed(self.random_state)

        # 1. Инициализация центроидов
        # Случайно выбираем n_clusters уникальных индексов из набора данных
        random_idx = np.random.choice(X.shape[0], self.n_clusters, replace=False)
        self.centroids = X[random_idx]

        for _ in range(self.max_iter):
            # 2. Назначение кластеров
            # Вычисляем расстояния от каждой точки до каждого центроида
            distances = self.compute_distances(X)
            labels = np.argmin(distances, axis=1)

            # 3. Пересчёт центроидов
            new_centroids = np.array([
                # Если кластер не пуст, новый центроид = среднее всех точек кластера
                X[labels == k].mean(axis=0) if len(X[labels == k]) > 0 else self.centroids[k]
                for k in range(self.n_clusters)
            ])

            # 4. Проверка сходимости
            shift = np.linalg.norm(self.centroids - new_centroids)
            self.centroids = new_centroids

            if shift < self.tol:
                break

        self.labels_ = labels
        return self

    def predict(self, X):
        """Предсказание кластеров для новых данных."""
        distances = self.compute_distances(X)
        return np.argmin(distances, axis=1) # возвращаем индексы ближайших центроидов

    def compute_distances(self, X):
        """Вычисление евклидовы расстояния от каждой точки X до каждого центроида."""
        #return np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
        X_squared = np.sum(X**2, axis=1).reshape(-1, 1)
        centroids_squared = np.sum(self.centroids**2, axis=1)
        distances = X_squared + centroids_squared - 2 * np.dot(X, self.centroids.T)
        return np.sqrt(np.maximum(distances, 0))

In [43]:
def intra_cluster_distance(X, labels):
    """Среднее внутрикластерное расстояние"""
    distances = [] # список для хранения средних расстояний по каждому кластеру
    
    for cluster in set(labels):
        if cluster == -1: # пропускаем шумовые точки
            continue
        points = X[labels == cluster] # оставляем только точки, принадлежащие текущему кластеру
        if len(points) < 2:
            continue
            
        # вычисляем центроид кластера — среднее по каждому признаку
        center = points.mean(axis=0)
        dist = np.linalg.norm(points - center, axis=1).mean()
        distances.append(dist)
        
    # если есть кластеры — возвращаем среднее по всем кластерам, иначе nan
    return np.mean(distances) if distances else np.nan 

In [44]:
def inter_cluster_distance(X, labels):
    """Среднее межкластерное расстояние"""
    centers = [] # список для хранения центроидов (средних точек) каждого кластера
    
    for cluster in set(labels):
        if cluster == -1: # игнорируем шумовые точки
            continue
        points = X[labels == cluster]
        centers.append(points.mean(axis=0))

    if len(centers) < 2: # если найден только один кластер или 0, межкластерное расстояние не определено
        return np.nan

    centers = np.array(centers)

    # Вычисление всех попарных расстояний между центроидами
    dists = []
    for i in range(len(centers)):
        for j in range(i + 1, len(centers)):
            dists.append(np.linalg.norm(centers[i] - centers[j]))

    # Возвращаем среднее арифметическое всех попарных расстояний между центроидами
    return np.mean(dists)

In [45]:
def evaluate_clustering(X, labels, pseudo_target):
    """Функция комплексной оценки качества кластеризации"""
    result = {}

    # проверка на 1 кластер
    if len(set(labels)) > 1:
        result["silhouette"] = silhouette_score(X, labels)
    else:
        result["silhouette"] = np.nan

    result["intra_mean"] = intra_cluster_distance(X, labels)
    result["inter_mean"] = inter_cluster_distance(X, labels)

    result["homogeneity"] = homogeneity_score(pseudo_target, labels)
    result["completeness"] = completeness_score(pseudo_target, labels)
    result["v_measure"] = v_measure_score(pseudo_target, labels)

    return result

In [46]:
taxi_results = []

# k-means

In [ ]:
# обучение для Taxi
def run_kmeans(k):
    model = MyKMeans(n_clusters=k, tol=1e-3, random_state=42)
    model.fit(X_taxi)
    labels = model.labels_

    metrics = evaluate_clustering(X_taxi, labels, pseudo_target_taxi)

    df = {
        "dataset": "taxi",
        "algo": "kmeans",
        "k": k,
        "max_iter": 50,
        "tol": 1e-3,
        "random_state": 42,
        "labels": labels,
        **metrics
    }
    
    return df

taxi_results = Parallel(n_jobs=4)( # запуск 4 параллельных процесса
    delayed(run_kmeans)(k) for k in range(2, 7)
)

In [47]:
fraud_results = []

In [48]:
mask = ~pseudo_target_fraud.isna()

X_fraud = X_fraud[mask]
pseudo_target_fraud = pseudo_target_fraud[mask]

In [ ]:
# обучение для Fraud
def run_kmeans(k):
    model = MyKMeans(n_clusters=k, tol=1e-3, random_state=42)
    model.fit(X_fraud)
    labels = model.labels_

    metrics = evaluate_clustering(X_fraud, labels, pseudo_target_fraud)

    df = {
        "dataset": "fraud",
        "algo": "kmeans",
        "k": k,
        "max_iter": 50,
        "tol": 1e-3,
        "random_state": 42,
        "labels": labels,
        **metrics
    }
    
    return df
    
fraud_results = Parallel(n_jobs=4)( # запуск 4 параллельных процесса
    delayed(run_kmeans)(k) for k in range(2, 7)
)

# Агломеративная кластеризация

In [50]:
def run_agg_clust(linkage, k, X, pseudo_target):
    model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
    labels = model.fit_predict(X)

    metrics = evaluate_clustering(X, labels, pseudo_target)

    df = {
        "dataset": X,
        "algo": "agg",
        "k": k,
        "linkage": linkage,
        "labels": labels,
        **metrics
    }
    
    return df

In [53]:
linkages = ['ward', 'complete', 'average', 'single']

In [ ]:
# Обучение для Taxi
for linkage in linkages:
    for k in range(4, 7):
        taxi_results.append(run_agg_clust(linkage, k, X_taxi, pseudo_target_taxi))

In [55]:
X_fraud_dense = X_fraud.toarray()

idx = np.random.choice(len(X_fraud_dense), 5000, replace=False)

X_small = X_fraud_dense[idx]
pseudo_small = pseudo_target_fraud.values[idx]

# Обучение для Fraud
for linkage in linkages:
    for k in range(4, 7):
        fraud_results.append(run_agg_clust(linkage, k, X_small, pseudo_small))

# DBSCAN

In [58]:
eps_values = [0.3, 0.5, 0.7]
min_samples_values = [3, 5, 10]

In [ ]:
# Обучение для Taxi
for eps in eps_values:
    for min_samples in min_samples_values:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels = model.fit_predict(X_taxi) # возвращение меток кластеров

        if len(set(labels)) <= 1: # если найден только один кластер или все точки — шум, пропускаем эту комбинацию
            continue

        metrics = evaluate_clustering(X_taxi, labels, pseudo_target_taxi)

        df = {
            "dataset": "taxi",
            "algo": "dbscan",
            "eps": eps,
            "min_samples": min_samples,
            "labels": labels,
            **metrics
        }
        
        print(df)
        
        taxi_results.append(df)

In [ ]:
# Обучение для Fraud
for eps in eps_values:
    for min_samples in min_samples_values:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels = model.fit_predict(X_fraud) # возвращение меток кластеров

        if len(set(labels)) <= 1: # если найден только один кластер или все точки — шум, пропускаем эту комбинацию
            continue

        metrics = evaluate_clustering(X_fraud, labels, pseudo_target_fraud)

        df = {
            "dataset": "fraud",
            "algo": "dbscan",
            "eps": eps,
            "min_samples": min_samples,
            "labels": labels,
            **metrics
        }
        
        print(df)
        
        fraud_results.append(df)

In [ ]:
taxi_results = pd.DataFrame(taxi_results)

In [ ]:
fraus_results = pd.DataFrame(fraud_results)

In [ ]:
# наилучший вариант кластеризации по метрике silhouette
best_row_taxi = taxi_results.sort_values(by="silhouette", ascending=False).iloc[0]
best_clusters_taxi = best_row_taxi["labels"]

In [ ]:
# наилучший вариант кластеризации по метрике silhouette
best_row_fraud = fraud_results.sort_values(by="silhouette", ascending=False).iloc[0]
best_clusters_fraud = best_row_fraud["labels"]

Ссылка на электронную таблицу


# Понижение размерности и 2D-визуализация

PCA

In [ ]:
class MyPCA:
    """Собственноручная реализация PCA"""
    def __init__(self, n_components=2):
        self.n_components = n_components

    def fit_transform(self, X):
        # центрирование
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_

        # ковариация
        cov_matrix = np.cov(X_centered, rowvar=False)

        # собственные значения/векторы
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # сортировка
        idx = np.argsort(eigenvalues)[::-1]
        eigenvectors = eigenvectors[:, idx]

        # проекция
        self.components_ = eigenvectors[:, :self.n_components]
        return X_centered @ self.components_

In [ ]:
pca = MyPCA(n_components=2)
X_taxi_pca = pca.fit_transform(X_taxi) # Обучение PCA

In [ ]:
X_fraud_pca = pca.fit_transform(X_fraud) # Обучение PCA

t-SNE

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_taxi_tsne = tsne.fit_transform(X_taxi) # Обучение t-SNE

In [ ]:
X_fraud_tsne = tsne.fit_transform(X_fraud) # Обучение t-SNE

UMAP

In [ ]:
import umap

umap_model = umap.UMAP(n_components=2, random_state=42)
X_taxi_umap = umap_model.fit_transform(X_taxi) # Обучение UMAP

In [ ]:
X_fraud_umap = umpa_model.fit_transform(X_fraud) # Обучение UMAP

Построение 2D-визуализации

In [ ]:
def plot_comparison(X_2d, pseudo_target, clusters, title):
    """Функция 2D-визуализации"""
    plt.figure(figsize=(12, 5))

    # Ground truth
    plt.subplot(1, 2, 1)
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=pseudo_target, s=5)
    plt.title(f"{title} — Ground Truth")

    # Clusters
    plt.subplot(1, 2, 2)
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, s=5)
    plt.title(f"{title} — Clusters")

    plt.show()

In [ ]:
# 2D-визуализация PCA-Taxi
plot_comparison(X_taxi_pca, pseudo_target_taxi, best_clusters_taxi, "PCA-Taxi") 

In [ ]:
# 2D-визуализация PCA-Fraud
plot_comparison(X_fraud_pca, pseudo_target_fraud, best_clusters_fraud, "PCA-Fraud")

In [ ]:
# 2D-визуализация t-SNE-Taxi
plot_comparison(X_taxi_tsne, pseudo_target_taxi, best_clusters_taxi, "t-SNE-Taxi")

In [ ]:
# 2D-визуализация t-SNE-Fraud
plot_comparison(X_fraud_tsne, pseudo_target_fraud, best_clusters_fraud, "t-SNE-Fraud")

In [ ]:
# 2D-визуализация UMAP-Taxi
plot_comparison(X_taxi_umap, pseudo_target_taxi, best_clusters_taxi, "UMAP-Taxi")

In [ ]:
# 2D-визуализация UMAP-Fraud
plot_comparison(X_fraud_umap, pseudo_target_fraud, best_clusters_fraud, "UMAP-Fraud")

Краткий анализ наблюдаемых различий

Метод PCA 

Метод t-SNE   

Метод UMAP  